# 5.2. Stage 5: Monthly Aggregation — Unique Vacancies

**Stage:** 5.2 of 5

**Purpose:** Aggregate the daily unique Parquet files produced by Stage 5.1 into monthly Parquet and JSON datasets.

**Input:** Completed daily Parquet files referenced by `process_df["rejoin_path"]`.

**Output:**
- Monthly Parquet files in `g.Config.STAGE5_MONTHLY_UNIQUE_PARQUET_PATH`
- Monthly JSON files in `g.Config.STAGE5_MONTHLY_UNIQUE_JSON_PATH`
- Monthly Ukraine-country JSON files in `g.Config.STAGE5_MONTHLY_UNIQUE_JSON_UA_PATH`

The months are derived automatically from the completed Stage 5.1 tracker rows. The `_ua_` monthly export intentionally retains records where `country == "Ukraine"`.

## 5.2.1. Load configuration and tracker

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys

import pandas as pd

from pipeline_bootstrap import configure_pipeline
configure_pipeline()
import general as g

g.clean_memory()

Load and sort the Stage 5.1 tracker. Only completed rows with a date-formatted `input_path` participate in monthly aggregation.

In [ ]:
process_df = pd.read_pickle(g.Config.STAGE5_PROCESS_UNIQUE_PATH)
process_df = process_df.sort_values(by="input_path").reset_index(drop=True)

process_df["input_date"] = pd.to_datetime(
    process_df["input_path"].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})", expand=False),
    errors="coerce",
)

invalid_date_count = process_df["input_date"].isna().sum()
if invalid_date_count:
    print(f"Skipping {invalid_date_count} tracker rows with invalid input dates.")

completed_process_df = process_df[
    process_df["rejoin_status"].eq("complete")
    & process_df["input_date"].notna()
].copy()

months_df = (
    completed_process_df["input_date"]
    .dt.to_period("M")
    .drop_duplicates()
    .sort_values()
    .to_frame(name="period")
    .reset_index(drop=True)
)
months_df["year"] = months_df["period"].dt.year
months_df["month"] = months_df["period"].dt.month

print(f"Completed daily files: {completed_process_df.shape[0]}")
print(f"Calendar months to aggregate: {months_df.shape[0]}")

## 5.2.2. Prepare output folders and schema

In [ ]:
g.check_folder_exists(g.Config.STAGE5_MONTHLY_UNIQUE_JSON_PATH)
g.check_folder_exists(g.Config.STAGE5_MONTHLY_UNIQUE_PARQUET_PATH)
g.check_folder_exists(g.Config.STAGE5_MONTHLY_UNIQUE_JSON_UA_PATH)

selected_columns = [
    "id", "date", "clean_title", "clean_desc", "min_salary", "max_salary",
    "currency", "salary_rate", "date_created", "date_expired", "desc_lang",
    "skill_ids", "skill_labels", "skill_labels_en", "job_type",
    "classified_code", "classified_title", "classified_title_clean",
    "extract_type", "esco_title", "esco_skills", "esco_id", "esco_code",
    "number_of_clicks", "region_original", "region", "city", "country",
    "latitude", "longitude",
]

## 5.2.3. Aggregate daily files by month

Each daily file is validated before use. Missing, unreadable, empty, or date-invalid files are reported and skipped. Missing schema columns are added as null columns through `reindex()`, allowing all valid daily files to be concatenated consistently.

In [ ]:
for _, month_row in months_df.iterrows():
    year = int(month_row["year"])
    month = int(month_row["month"])
    period = month_row["period"]
    print(f"Month: {year}-{month}")

    month_tracker = completed_process_df[
        completed_process_df["input_date"].dt.to_period("M").eq(period)
    ]
    daily_frames = []

    for _, process_row in month_tracker.iterrows():
        rejoin_path = process_row["rejoin_path"]
        if pd.isna(rejoin_path) or not str(rejoin_path).strip():
            print(f"Skipping {process_row['input_path']}: missing rejoin path.")
            continue
        if not os.path.isfile(rejoin_path):
            print(f"Skipping {process_row['input_path']}: file not found: {rejoin_path}")
            continue

        try:
            daily_data = pd.read_parquet(rejoin_path)
        except Exception as error:
            print(f"Skipping {process_row['input_path']}: cannot read Parquet: {error}")
            continue

        if daily_data.empty:
            print(f"Skipping {process_row['input_path']}: daily file is empty.")
            continue
        if "date" not in daily_data.columns:
            print(f"Skipping {process_row['input_path']}: required column 'date' is missing.")
            continue

        daily_data["date"] = pd.to_datetime(daily_data["date"], errors="coerce")
        valid_dates = daily_data["date"].dropna()
        if valid_dates.empty:
            print(f"Skipping {process_row['input_path']}: column 'date' has no valid values.")
            continue
        if not valid_dates.dt.to_period("M").eq(period).all():
            print(f"Skipping {process_row['input_path']}: records do not belong to {period}.")
            continue

        missing_columns = [
            column for column in selected_columns if column not in daily_data.columns
        ]
        if missing_columns:
            print(
                f"Warning for {process_row['input_path']}: "
                f"adding missing columns as null: {missing_columns}"
            )

        daily_frames.append(daily_data.reindex(columns=selected_columns).copy())

    if not daily_frames:
        print(f"Month {year}-{month}: no valid daily files; no outputs created.")
        continue

    month_data = pd.concat(daily_frames, ignore_index=True)
    output_name = f"{year}-{month}"

    json_path = os.path.join(
        g.Config.STAGE5_MONTHLY_UNIQUE_JSON_PATH,
        output_name + ".json",
    )
    month_data.to_json(json_path, orient="records", date_format="iso")

    parquet_path = os.path.join(
        g.Config.STAGE5_MONTHLY_UNIQUE_PARQUET_PATH,
        output_name + ".parquet",
    )
    month_data.to_parquet(parquet_path, index=False)

    ua_json_path = os.path.join(
        g.Config.STAGE5_MONTHLY_UNIQUE_JSON_UA_PATH,
        output_name + ".json",
    )
    month_data[month_data["country"] == "Ukraine"].to_json(
        ua_json_path,
        orient="records",
        force_ascii=False,
        date_format="iso",
    )

    print(
        f"Month {year}-{month}: complete — "
        f"{len(daily_frames)} daily files, {month_data.shape[0]} rows."
    )

print("All done!")

---
© 2026 Yurii Kleban, Britta Rude. All rights reserved.